# Oil & Gas Production Forecast Refresh
### Microsoft Fabric + Local Demo Notebook

This portfolio-safe notebook supports both Microsoft Fabric and local execution modes.

- `RUN_MODE = "fabric"` reads registered Lakehouse Delta tables and writes Gold Delta tables.
- `RUN_MODE = "local"` reads `ProductionDaily.csv` and `WellInfo.csv` from the notebook folder and writes Power BI-ready CSV files locally.

The cleaning, downtime detection, ARPS fitting, baseline uptime calculation, date dimension creation, and fit-quality logic are shared between both modes.


> **Portfolio note:** This notebook is designed for synthetic or anonymised demonstration data. It contains no credentials, workspace IDs, SQL endpoints, or organisation-specific connection strings.


## 1. Configuration

For a local demonstration, leave `RUN_MODE = "local"`.

For Microsoft Fabric execution, change it to `"fabric"` and update the registered table names if necessary.


In [ ]:
from pathlib import Path

RUN_MODE = "local"   # "local" or "fabric"

# ---------- Fabric source table names ----------
SOURCE_PRODUCTION = "ProductionDaily"
SOURCE_WELLINFO   = "WellInfo"

# ---------- Fabric Gold table names ----------
GOLD_PARAMETERS = "gold_well_parameters"
GOLD_DATE_DIM   = "gold_date_dim"
GOLD_ACTUALS    = "gold_actuals"
GOLD_FIT_LOG    = "gold_fit_quality_log"

# ---------- Local input files ----------
BASE_DIR = Path.cwd()
LOCAL_PRODUCTION_FILE = BASE_DIR / "ProductionDaily.csv"
LOCAL_WELLINFO_FILE   = BASE_DIR / "WellInfo.csv"

# ---------- Local output folder/files ----------
LOCAL_OUTPUT_DIR = BASE_DIR / "demo_outputs"

LOCAL_PARAMETERS_FILE = LOCAL_OUTPUT_DIR / "WellParameters.csv"
LOCAL_DATE_DIM_FILE   = LOCAL_OUTPUT_DIR / "DateDim.csv"
LOCAL_ACTUALS_FILE    = LOCAL_OUTPUT_DIR / "Actuals.csv"

# The fit-quality log is written separately so model-monitoring history can be retained.
LOCAL_FIT_LOG_FILE    = LOCAL_OUTPUT_DIR / "FitQualityLog.csv"

# ---------- Model settings ----------
OIL_PRODUCT_CODE = "OIL"

FORECAST_EXTENSION_DAYS = 365
DOWNTIME_WINDOW = 15
DOWNTIME_THRESHOLD = 0.6
MIN_R2_ALERT = 0.50

print("RUN_MODE:", RUN_MODE)
print("Working folder:", BASE_DIR)


## 2. Imports and run timestamp


In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

RUN_TIMESTAMP = datetime.utcnow()

if RUN_MODE not in {"local", "fabric"}:
    raise ValueError("RUN_MODE must be either 'local' or 'fabric'.")

if RUN_MODE == "fabric":
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

print("Run started:", RUN_TIMESTAMP)


## 3. Read and validate source data

The production source must contain:

- `asset_key`
- `date_key`
- `product_key`
- `production_qty`

The well-reference source must contain:

- a well identifier (`well_id` or `asset_key`)
- `field_name`
- `producing_method`

If the local WellInfo file uses `asset_key` instead of `well_id`, this notebook renames it automatically for the Gold parameter table.


In [ ]:
if RUN_MODE == "fabric":
    daily = spark.read.table(SOURCE_PRODUCTION).toPandas()
    wells = spark.read.table(SOURCE_WELLINFO).toPandas()

else:
    if not LOCAL_PRODUCTION_FILE.exists():
        raise FileNotFoundError(
            f"Missing {LOCAL_PRODUCTION_FILE.name}. "
            f"Place it in: {BASE_DIR}"
        )

    if not LOCAL_WELLINFO_FILE.exists():
        raise FileNotFoundError(
            f"Missing {LOCAL_WELLINFO_FILE.name}. "
            f"Place it in: {BASE_DIR}"
        )

    daily = pd.read_csv(LOCAL_PRODUCTION_FILE)
    wells = pd.read_csv(LOCAL_WELLINFO_FILE)

production_required = {
    "asset_key",
    "date_key",
    "product_key",
    "production_qty",
}

missing_production = production_required.difference(daily.columns)
if missing_production:
    raise ValueError(
        "ProductionDaily is missing required columns: "
        f"{sorted(missing_production)}"
    )

# Accept either well_id or asset_key as the well-reference key.
if "well_id" not in wells.columns:
    if "asset_key" in wells.columns:
        wells = wells.rename(columns={"asset_key": "well_id"})
    else:
        raise ValueError(
            "WellInfo must contain either 'well_id' or 'asset_key'."
        )

well_required = {
    "well_id",
    "field_name",
    "producing_method",
}

missing_well = well_required.difference(wells.columns)
if missing_well:
    raise ValueError(
        "WellInfo is missing required columns: "
        f"{sorted(missing_well)}"
    )

daily["date_key"] = pd.to_datetime(
    daily["date_key"],
    errors="coerce"
)

daily["production_qty"] = pd.to_numeric(
    daily["production_qty"],
    errors="coerce"
)

daily = daily.dropna(
    subset=[
        "asset_key",
        "date_key",
        "product_key",
        "production_qty",
    ]
).copy()

oil = daily[
    daily["product_key"] == OIL_PRODUCT_CODE
].copy()

if oil.empty:
    raise ValueError(
        f"No rows found where product_key == '{OIL_PRODUCT_CODE}'."
    )

print(
    f"Loaded {len(oil):,} oil production rows "
    f"across {oil.asset_key.nunique()} wells"
)
print(
    "Date range:",
    oil.date_key.min().date(),
    "to",
    oil.date_key.max().date()
)
print("WellInfo rows:", len(wells))


## 4. Downtime detection and ARPS fitting

Short, sharp production drops are treated as operational downtime rather than reservoir decline.

The ARPS fit is performed only on non-downtime observations. Baseline uptime is then calculated from actual production relative to fitted potential production.


In [ ]:
def flag_downtime(
    q,
    window=DOWNTIME_WINDOW,
    threshold=DOWNTIME_THRESHOLD
):
    """Flag days well below the local rolling median."""
    s = pd.Series(q)
    med = s.rolling(
        window=window,
        center=True,
        min_periods=3
    ).median()

    return (
        s < threshold * med
    ).fillna(False).to_numpy()


def arps(t, qi, di, b):
    """ARPS hyperbolic decline.

    q(t) = qi / (1 + b*di*t)^(1/b)

    qi = initial rate
    di = nominal daily decline rate
    b  = decline exponent
    """
    return qi / np.power(
        1 + b * di * t,
        1 / b
    )


def fit_well(g):
    g = (
        g.sort_values("date_key")
        .reset_index(drop=True)
        .copy()
    )

    g["day_index"] = np.arange(len(g))

    q = g["production_qty"].to_numpy(dtype=float)
    t = g["day_index"].to_numpy(dtype=float)

    downtime = flag_downtime(q)

    clean_t = t[~downtime]
    clean_q = q[~downtime]

    if len(clean_t) < 30:
        return None, None

    qi_guess = np.percentile(
        clean_q[:30],
        90
    )

    params, _ = curve_fit(
        arps,
        clean_t,
        clean_q,
        p0=[qi_guess, 0.001, 0.6],
        bounds=(
            [0, 1e-6, 0.01],
            [qi_guess * 3, 0.05, 2.0]
        ),
        maxfev=10000,
    )

    qi, di, b = params

    fitted = arps(
        clean_t,
        qi,
        di,
        b
    )

    denominator = np.sum(
        (clean_q - clean_q.mean()) ** 2
    )

    if denominator == 0:
        r2 = np.nan
    else:
        r2 = (
            1
            - np.sum(
                (clean_q - fitted) ** 2
            )
            / denominator
        )

    potential = arps(
        t,
        qi,
        di,
        b
    )

    uptime = float(
        np.clip(
            q.sum() / potential.sum(),
            0.5,
            1.0
        )
    )

    parameter_row = {
        "qi": round(float(qi), 2),
        "di": round(float(di), 8),
        "b": round(float(b), 4),
        "baseline_uptime": round(uptime, 4),
        "downtime_loss_pct": round(
            (1 - uptime) * 100,
            2
        ),
        "fit_r2": (
            round(float(r2), 4)
            if not np.isnan(r2)
            else np.nan
        ),
        "history_days": int(len(g)),
    }

    actuals = g[
        [
            "asset_key",
            "date_key",
            "day_index",
            "production_qty",
        ]
    ].copy()

    return parameter_row, actuals


## 5. Fit every oil-producing well


In [ ]:
param_rows = []
actual_frames = []

for well, g in oil.groupby("asset_key"):

    try:
        fit, actuals_g = fit_well(g)

    except Exception as exc:
        print(
            f"  SKIPPED {well}: fitting error - {exc}"
        )
        continue

    if fit is None:
        print(
            f"  SKIPPED {well}: insufficient clean history"
        )
        continue

    param_rows.append(
        {
            "well_id": well,
            **fit
        }
    )

    actual_frames.append(
        actuals_g
    )

if not param_rows:
    raise RuntimeError(
        "No wells were successfully fitted."
    )

params_df = pd.DataFrame(
    param_rows
).merge(
    wells[
        [
            "well_id",
            "field_name",
            "producing_method",
        ]
    ].drop_duplicates("well_id"),
    on="well_id",
    how="left"
)

params_df["refreshed_at"] = RUN_TIMESTAMP

actuals_df = pd.concat(
    actual_frames,
    ignore_index=True
)

print(
    f"Fitted {len(params_df)} wells"
)

print(
    "Mean fit R2:",
    round(
        params_df["fit_r2"].mean(),
        3
    )
)

degraded = params_df[
    params_df["fit_r2"] < MIN_R2_ALERT
]

if len(degraded):

    print(
        f"\nWARNING - {len(degraded)} well(s) "
        f"below R2 {MIN_R2_ALERT}:"
    )

    print(
        degraded[
            [
                "well_id",
                "fit_r2"
            ]
        ].to_string(index=False)
    )

    print(
        "\nA low R2 can reflect a real production event "
        "such as a workover, choke change, or completion change."
    )


## 6. Build the date dimension

The date dimension extends 365 days beyond the historical period so Power BI can evaluate the ARPS equation into the future while the actual-production line naturally stops at the end of history.


In [ ]:
max_day = int(
    actuals_df["day_index"].max()
)

start_date = (
    actuals_df["date_key"].min()
)

total_days = (
    max_day
    + 1
    + FORECAST_EXTENSION_DAYS
)

date_dim = pd.DataFrame({
    "day_index": np.arange(
        total_days
    ),
    "date_key": pd.date_range(
        start_date,
        periods=total_days,
        freq="D"
    ),
})

date_dim["is_forecast"] = (
    date_dim["day_index"] > max_day
)

print(
    f"Date dim: {len(date_dim):,} rows"
)

print(
    "History to:",
    date_dim[
        ~date_dim["is_forecast"]
    ]["date_key"].max().date()
)

print(
    "Forecast to:",
    date_dim["date_key"].max().date()
)


## 7. Write Gold outputs

- In Fabric mode, the notebook writes Delta tables.
- In local mode, it writes Power BI-ready CSV files to `demo_outputs`.

The fit-quality log is appended locally when the file already exists, preserving run history.


In [ ]:
fit_log = params_df[
    [
        "well_id",
        "fit_r2",
        "qi",
        "di",
        "b",
        "downtime_loss_pct",
        "refreshed_at",
    ]
].copy()


if RUN_MODE == "fabric":

    def write_delta(
        pdf,
        table_name,
        mode="overwrite"
    ):
        sdf = spark.createDataFrame(pdf)

        (
            sdf.write
            .format("delta")
            .mode(mode)
            .option(
                "overwriteSchema",
                "true"
            )
            .saveAsTable(
                table_name
            )
        )

        print(
            f"  wrote {table_name}: "
            f"{len(pdf):,} rows ({mode})"
        )


    write_delta(
        params_df,
        GOLD_PARAMETERS
    )

    write_delta(
        date_dim,
        GOLD_DATE_DIM
    )

    write_delta(
        actuals_df,
        GOLD_ACTUALS
    )

    write_delta(
        fit_log,
        GOLD_FIT_LOG,
        mode="append"
    )


else:

    LOCAL_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    params_df.to_csv(
        LOCAL_PARAMETERS_FILE,
        index=False
    )

    date_dim.to_csv(
        LOCAL_DATE_DIM_FILE,
        index=False
    )

    actuals_df.to_csv(
        LOCAL_ACTUALS_FILE,
        index=False
    )

    if LOCAL_FIT_LOG_FILE.exists():

        existing_log = pd.read_csv(
            LOCAL_FIT_LOG_FILE
        )

        combined_log = pd.concat(
            [
                existing_log,
                fit_log
            ],
            ignore_index=True
        )

        combined_log.to_csv(
            LOCAL_FIT_LOG_FILE,
            index=False
        )

    else:

        fit_log.to_csv(
            LOCAL_FIT_LOG_FILE,
            index=False
        )

    print(
        "\nLocal demo files written to:"
    )
    print(LOCAL_OUTPUT_DIR)

    print(
        "\nFiles:"
    )
    print(
        " ",
        LOCAL_PARAMETERS_FILE.name
    )
    print(
        " ",
        LOCAL_DATE_DIM_FILE.name
    )
    print(
        " ",
        LOCAL_ACTUALS_FILE.name
    )
    print(
        " ",
        LOCAL_FIT_LOG_FILE.name
    )


print(
    "\nRefresh complete at",
    datetime.utcnow()
)


## 8. Final validation


In [ ]:
print(
    "Well parameter rows:",
    len(params_df)
)

print(
    "Actual rows:",
    len(actuals_df)
)

print(
    "Date dimension rows:",
    len(date_dim)
)

print(
    "Fit log rows this run:",
    len(fit_log)
)

print(
    "Unique fitted wells:",
    params_df["well_id"].nunique()
)

if RUN_MODE == "local":
    expected_files = [
        LOCAL_PARAMETERS_FILE,
        LOCAL_DATE_DIM_FILE,
        LOCAL_ACTUALS_FILE,
        LOCAL_FIT_LOG_FILE,
    ]

    for file_path in expected_files:
        print(
            file_path.name,
            "exists:",
            file_path.exists()
        )


## 9. Power BI handoff

For the local fallback demo, import:

- `WellParameters.csv`
- `DateDim.csv`
- `Actuals.csv`

The fit-quality log supports the model-transparency / monitoring page.

The Power BI What-If measures evaluate the ARPS equation from `qi`, `di`, and `b` at query time. Neutral What-If settings should reproduce the fitted base curve.
